# 03 — Biometric GAMs (Fig 3, S6–S13)

Population-level GAM fits of the five core biometrics (RHR, HRV, RR, skin
temp, blood O₂) on cycle day × cycle length × age, with cardiovascular,
sleep, season, and BMI covariates. Reproduces Fig 3 plus seven supplementary
figures.

Mirrors the source notebook (`figure_generator_revision.ipynb`) cells 1, 2,
4, 8, 9, 68 (setup), 69, 70 (fit + metrics), 72, 73 (Fig 3), 75, 76
(magnitude range contrasts).

Requires R 4.x with `mgcv` installed and `rpy2` on `PYTHONPATH`. Fitted
models are cached to `models/pct_*.rds`. To refit from scratch, delete the
files in `models/` first.

In [ ]:
try:
    import rpy2  # noqa: F401
    from rpy2.robjects.packages import importr
    importr('mgcv')
except Exception as e:
    raise RuntimeError(
        "Notebook 03 requires rpy2 and R with the `mgcv` package. "
        "Install via `pip install -e .[r]` and ensure R 4.x is on PATH "
        "with mgcv installed (`install.packages('mgcv')`)."
    ) from e

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from menstrual_cycle_analysis import (
    CycleLengthAnalyses,
    PhysioMethods,
    config,
    load_paper_data,
)
from menstrual_cycle_analysis.r_utils import load_gam_models, save_gam_models

np.random.seed(0)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Setup (mirrors source cells 1, 2, 4, 8, 9, 68)

Cell 68 in the source builds a `PhysioBehaviorAnalyses` instance whose
constructor calls a chain of `pm.add_*` helpers as a side effect on `PM`.
We invoke those helpers directly here — same end state, no detour through
the PBA constructor.

In [ ]:
day_df, CBM = load_paper_data()

CBM.add_sleep_behaviors('user')
CBM.add_workout_behaviors('user')
CBM.add_sleep_behaviors('cycle')
CBM.add_workout_behaviors('cycle')

cla = CycleLengthAnalyses(CBM=CBM)

PM = PhysioMethods(cbm=CBM)
PM.get_reference_table()
PM.process_physio_data(overwrite=True)

PM.add_wo_data()
PM.add_sleep_2_ref()
PM.add_wo_2_ref()
PM.add_acute_chronic_behaviors_data()
PM.add_acute_chronic_behaviors_ref()
PM.add_seasonalilty_2_ref()

## Fit GAM models (mirrors source cell 69)

Five `mgcv::bam` models, one per core biometric, with AR(1) within
subject-cycle. Each fit takes ~5–30 minutes; the loop below caches fitted
models and the metrics table to `models/` so subsequent runs are fast.

In [ ]:
PM.gam_additional_covariates = config.GAM_ADDITIONAL_COVARIATES

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = config.MODELS_DIR / 'gam_models_metrics.parquet'
biometrics = [f'pct_{vv}' for vv in PM.CORE_BIOMETRICS]
cache_complete = (
    all((config.MODELS_DIR / f'{b}.rds').exists() for b in biometrics)
    and metrics_path.exists()
)

if cache_complete:
    print('Cache hit: loading fitted GAMs from models/')
    PM.gam_models = {}
    load_gam_models(PM, config.MODELS_DIR, keys=biometrics)
    PM.gam_models_metrics = pd.read_parquet(metrics_path)
    # Rebuild gam_data so downstream prediction helpers can reference covariate means.
    PM.prep_data_gam_cycle_model(prefix='pct', additional_covariates=PM.gam_additional_covariates)
else:
    PM.fit_gam_models(
        additional_covariates=PM.gam_additional_covariates,
        overwrite=False,
    )
    save_gam_models(PM, config.MODELS_DIR, overwrite=False)
    PM.gam_models_metrics.to_parquet(metrics_path)

In [ ]:
print(PM.gam_models_metrics)

## Figure 3 — population GAM curves by age and cycle length (mirrors source cells 72, 73)

In [ ]:
PM.gam_sim_data = PM.get_gam_cl_age_sim_data()

In [ ]:
f, axes = PM.plot_gam_biometrics_cl_age(gam_data=PM.gam_sim_data)

fig_path_svg = config.FIGURES_DIR / 'fig3_gam_biometrics_cl_age.svg'
fig_path_png = config.FIGURES_DIR / 'fig3_gam_biometrics_cl_age.png'
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
f.savefig(fig_path_svg, bbox_inches='tight')
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_svg.name, '+', fig_path_png.name)

## Magnitude range contrasts (mirrors source cells 75, 76)

In [ ]:
for biometric in PM.CORE_BIOMETRICS:
    print("-" * 80)
    col = f'pct_{biometric}'
    PM.print_gam_age_contrast(biometric=col)

In [ ]:
for biometric in PM.CORE_BIOMETRICS:
    print("-" * 80)
    col = f'pct_{biometric}'
    PM.print_gam_cl_contrast(biometric=col)

## Figure S6 — user-level biometrics vs age (mirrors source cell 127)

In [ ]:
f, axes = PM.plot_user_level_biometrics_vs_x(prefix='b', x_var='age_b')

fig_path_svg = config.FIGURES_DIR / 'figS6_b_user_level_biometrics_vs_age.svg'
fig_path_png = config.FIGURES_DIR / 'figS6_b_user_level_biometrics_vs_age.png'
f.savefig(fig_path_svg, bbox_inches='tight')
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_svg.name, '+', fig_path_png.name)

## Figure S7 — user-level biometrics vs BMI (mirrors source cell 130)

In [ ]:
f, axes = PM.plot_user_level_biometrics_vs_x(prefix='b', x_var='BMI_b2')

fig_path_svg = config.FIGURES_DIR / 'figS7_b_user_level_biometrics_vs_BMI.svg'
fig_path_png = config.FIGURES_DIR / 'figS7_b_user_level_biometrics_vs_BMI.png'
f.savefig(fig_path_svg, bbox_inches='tight')
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_svg.name, '+', fig_path_png.name)

## Figure S13 — user-level biometrics vs mean sleep duration (mirrors source cell 171)

In [ ]:
f, axes = PM.plot_user_level_biometrics_vs_x(prefix='b', x_var='sl_dur_mean_bin')
axes[-1, 0].set_xticklabels(CBM.sl_dur_labels, rotation=45)
axes[-1, 1].set_xticklabels(CBM.sl_dur_labels, rotation=45)

fig_path_svg = config.FIGURES_DIR / 'figS13_b_user_level_biometrics_vs_sl_dur_mean_bin.svg'
fig_path_png = config.FIGURES_DIR / 'figS13_b_user_level_biometrics_vs_sl_dur_mean_bin.png'
f.savefig(fig_path_svg, bbox_inches='tight')
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_svg.name, '+', fig_path_png.name)

## Figure S8 — empirical biometrics × age × cycle length (mirrors source cell 132)

In [ ]:
f, axes = PM.plot_biometrics_cl_age(prefix='pct')

fig_path_svg = config.FIGURES_DIR / 'figS8_pct_biometrics_cl_age_data.svg'
fig_path_png = config.FIGURES_DIR / 'figS8_pct_biometrics_cl_age_data.png'
f.savefig(fig_path_svg, bbox_inches='tight')
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_svg.name, '+', fig_path_png.name)

## Figure S10 — critical-value heatmaps from GAM surfaces (mirrors source cells 142-150)

Dense-grid simulation through the fitted GAMs, then locate the per-biometric
min, max, and zero-crossing day across age × cycle-length. The PNG is the only
saved format here (heatmap policy in CLAUDE.md).

In [ ]:
gam_sim_data = PM.get_gam_sim_data_for_vars(
    var_grid_dict={'d': np.arange(0, 35, 0.25), 'age': np.arange(18, 51, 1), 'cl': np.arange(21, 36, 1)},
    se_fit=False,
)

In [ ]:
out_cwt = PM.get_gam_critical_values_cwt(gam_sim_data, window_w=5)

In [ ]:
out_simp = PM.get_gam_critical_values_simple(gam_sim_data)

In [ ]:
cl1 = 24
cl2 = 34
age = 32
print("=" * 80)
print(f"CL Contrast at age={age}")

for vv in PM.CORE_BIOMETRICS:
    print("-" * 80)
    A = out_simp[vv].min_loc.unstack().T
    print(f"{vv} min locs {A.loc[age].round()[[cl1, cl2]]}")
    print()
    A = out_simp[vv].max_loc.unstack().T
    print(f"{vv} max locs {A.loc[age].round()[[cl1, cl2]]}")
    print()
print("=" * 80)

In [ ]:
age1 = 24
age2 = 44

for cl in [28]:
    print("=" * 40)
    print(f"Age Contrast at cl={cl}")
    for vv in PM.CORE_BIOMETRICS:
        print("-" * 40)
        A = out_simp[vv].min_loc.unstack().T
        print(f"{vv} min locs {A[cl].round().loc[[age1, age2]]}")
        print()
        A = out_simp[vv].max_loc.unstack().T
        print(f"{vv} max locs {A[cl].round().loc[[age1, age2]]}")
        print()
    print("=" * 40)

In [ ]:
f, _ = PM.plot_biometrics_critical_values_heatmaps2(out_simp)

fig_path_png = config.FIGURES_DIR / 'figS10_critical_values_heatmaps.png'
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_png.name)

## Figure S11 — biometric magnitude range × CL × age (mirrors source cells 152-165)

GAM-implied range (max - min over cycle days) and a parallel within-cycle GEE
fit on the per-cycle delta, both as age × CL heatmaps. PNG only (heatmap policy).

In [ ]:
model_biom_delta_x_cl_age = PM.model_biom_delta_x_cl_age()

In [ ]:
model_biom_delta_x_cl_age['pct_RHR']

In [ ]:
gam_biom_delta_x_cl_age = PM.gam_biom_delta_x_cl_age()

In [ ]:
f, axes = PM.plot_max_min_biometric_cycle_age_cl_variation_gam(
    prefix='pct',
    gam_results=gam_biom_delta_x_cl_age,
    within_cycle_results=model_biom_delta_x_cl_age,
)

fig_path_png = config.FIGURES_DIR / 'figS11_pct_delta_mag_cl_age.png'
f.savefig(fig_path_png, bbox_inches='tight', dpi=config.FIGURE_DPI)
print('Saved', fig_path_png.name)